### 工作進度  
* 【置頂】**筆記內容架構**與**量化技術分析系統**相關資訊請參閱[260315筆記.ipynb](https://github.com/yilintung/StockInvestmentNotebook/blob/main/260315%E7%AD%86%E8%A8%98.ipynb)之「工作進度」。
* 本日延續[260319筆記.ipynb](https://github.com/yilintung/StockInvestmentNotebook/blob/main/260319%E7%AD%86%E8%A8%98.ipynb)的「技術開發」。  

In [1]:
import os
import pandas as pd
import numpy as np
import datetime
import sqlite3
import io
import base64
import matplotlib.pyplot as plt
import matplotlib
import mplfinance as mpf
import markdown

import json

from talib.abstract import *
from dotenv import load_dotenv, find_dotenv
from PIL import Image
from IPython.core.display import HTML

from typing import Any, Dict, List, Optional, Tuple
from openai import OpenAI

In [2]:
##### 【公用函式】 來源 ： https://www.cnblogs.com/Rosaany/p/15561918.html #####
def get_monday_to_sunday(today, weekly=0):
    last = weekly * 7
    today = datetime.datetime.strptime(str(today), "%Y-%m-%d")
    monday = datetime.datetime.strftime(today - datetime.timedelta(today.weekday() - last), "%Y-%m-%d")
    monday_ = datetime.datetime.strptime(monday, "%Y-%m-%d")
    sunday = datetime.datetime.strftime(monday_ + datetime.timedelta(monday_.weekday() + 6), "%Y-%m-%d")
    return monday, sunday

###### 【內部函式】 當日Ｋ價格資料為零時之修正函式 ######
def correcting_zero_price_issue( daily_price_df, debug = False) :
    # 開啟寫時複製(Copy-on-Write)
    copy_on_write = pd.options.mode.copy_on_write
    pd.options.mode.copy_on_write = True
    
    # 找出開盤價、最高價、收盤價與最低價皆為0的價格資料
    zero_prices_df =  daily_price_df[(daily_price_df['Open'] == 0.0) & (daily_price_df['High'] == 0.0) & (daily_price_df['Low'] == 0.0) & (daily_price_df['Close'] == 0.0)]
    if zero_prices_df.empty is False :
        zero_prices_idx = zero_prices_df.index
        df_first_idx    = daily_price_df.iloc[0].name
        for idx in zero_prices_idx :
            if (idx - 1) >= df_first_idx :
                # 當開盤價、最高價、收盤價與最低價皆為0時，會用前一個交易日的收盤價來做修正
                prev_close_price = daily_price_df.loc[idx-1]['Close']
                prev_stock_id    = daily_price_df.loc[idx-1]['StockID']
                curr_stock_id    = daily_price_df.loc[idx]['StockID']
                if debug is True :
                    print('ＤＥＢＵＧ ： 〈代碼：{}，日期：{}〉  修改前：開 ＝ {} 高 ＝ {} 低 ＝ {} 收 ＝ {}'.format(curr_stock_id,daily_price_df.loc[idx]['Date'],daily_price_df.loc[idx,'Open'],daily_price_df.loc[idx,'High'],daily_price_df.loc[idx,'Low'],daily_price_df.loc[idx,'Close']), end='')
                daily_price_df.loc[idx,'Open']  = prev_close_price
                daily_price_df.loc[idx,'High']  = prev_close_price
                daily_price_df.loc[idx,'Low']   = prev_close_price
                daily_price_df.loc[idx,'Close'] = prev_close_price
                if debug is True :
                    print(' ， 修改後：開 ＝ {} 高 ＝ {} 低 ＝ {} 收 ＝ {}'.format(daily_price_df.loc[idx,'Open'],daily_price_df.loc[idx,'High'],daily_price_df.loc[idx,'Low'],daily_price_df.loc[idx,'Close']))
            else :
                pass
    
    # 還原初始狀態
    pd.options.mode.copy_on_write = copy_on_write

###### 【公用函式】 來源 ： https://stackoverflow.com/questions/73604477/i-am-trying-to-crop-an-image-to-remove-extra-space-in-python #####
def get_first_last(mask, axis: int):
    """ Find the first and last index of non-zero values along an axis in `mask` """
    mask_axis = np.argmax(mask, axis=axis) > 0
    a = np.argmax(mask_axis)
    b = len(mask_axis) - np.argmax(mask_axis[::-1])
    return int(a), int(b)

def crop_borders(img, crop_color):
    np_img = np.array(img)
    mask = (np_img != crop_color)[..., 0]   # compute a mask
    x0, x1 = get_first_last(mask, 0)        # find boundaries along x axis
    y0, y1 = get_first_last(mask, 1)        # find boundaries along y axis
    return img.crop((x0, y0, x1, y1)) 

def result_to_dataframe(result: dict) -> pd.DataFrame:
    df = pd.DataFrame(
        {
            "解盤內容": result
        }
    )
    df.index.name = "技術分析工具"
    return df
def display_result( result) :
    result_df = result_to_dataframe( result)
    result_md   = result_df.to_markdown(tablefmt="grid")
    result_html = markdown.markdown(result_md, extensions=['markdown_grid_tables:GridTableExtension'])
    display(HTML(result_html))

In [3]:
import json
from typing import Tuple
from openai import OpenAI


# =========================
# PROMPT BUILDERS
# =========================


def build_system_prompt(instrument: dict) -> str:
    return f"""
你是一位技術分析師，使用「看圖優先」。

【核心原則】
- 由長到短（長→中→短）
- 最終必須整合成單一結論
- 不可暴露技術欄位名稱
- 型態可以不存在，不可硬湊
- 可以用數據輔助確認，但輸出要像人類看圖分析

【MACD輸出規範】
只允許：DIF線、MACD線、OSC柱體
禁止：macd / macdsignal / macdhist / 訊號線

【標的】
{json.dumps(instrument, ensure_ascii=False)}
""".strip()



def build_phase1_prompt(stock_id: str) -> str:
    return f"""
你會收到三張圖：
1. 長週期（240）
2. 中週期（120）
3. 短週期（60）

請由長 → 中 → 短進行分析。

--------------------------------------------------

【型態優先決策（強制）】

請優先識別技術型態，而不是先下趨勢結論。

順序：
1. 先判斷是否存在型態
2. 已成立優先
3. 成形中次之
4. 沒有才用趨勢（多頭 / 空頭 / 盤整）

--------------------------------------------------

【最近型態優先】

只選擇「最近、仍影響當前價格」的型態。

--------------------------------------------------

【跨週期一致性（強制）】

若長或中週期已出現明確反轉型態（如雙重底）：

- 中、短週期不可改判為 box 覆蓋主型態
- 區間只能視為型態中的整理
- structure_type 必須維持一致主軸

--------------------------------------------------

【區間不得覆蓋底型（強制）】

若出現：
- 兩個相近低點
- 中間反彈
- 第二低點未破前低
- 價格重新走高

→ 必須使用 double_bottom
→ 不可判為 box

--------------------------------------------------

【型態輸出】

- structure_type：主型態或 none（僅內部使用）
- structure_range：型態時間
- note：補充

--------------------------------------------------

【型態時間】

若有型態：
- 提供起訖日期（可為自然日期字串）

--------------------------------------------------

【支撐壓力】

來源僅限：
- 高低點
- 均線
- 區間

--------------------------------------------------

【need_price_data】

只有以下才 YES：
- 支撐壓力不明
- KD不清
- 指標影響判斷

--------------------------------------------------

【輸出 JSON】

{{
  "stock_id": "{stock_id}",
  "long_view": {{
    "structure_type": "...",
    "structure_range": "...",
    "note": "..."
  }},
  "mid_view": {{
    "structure_type": "...",
    "structure_range": "...",
    "note": "..."
  }},
  "short_view": {{
    "structure_type": "...",
    "structure_range": "...",
    "note": "..."
  }},
  "need_price_data": "YES | NO",
  "need_price_data_reason": ""
}}
""".strip()



def build_phase2_prompt(stock_id: str, structure: dict, instrument: dict, price_data=None) -> str:
    prompt = f"""
請分析 {stock_id}。

【結構】
{json.dumps(structure, ensure_ascii=False)}

【標的資訊】
{json.dumps(instrument, ensure_ascii=False)}

--------------------------------------------------

【型態必須承接 structure（強制）】

- 不可忽略 structure 中的型態
- 不可改寫或降級型態
- structure 是最終型態判斷的唯一來源，不是參考來源

--------------------------------------------------

【型態名稱必須一致（強制）】

型態名稱必須與 structure 中的 structure_type 完全一致，不可自行替換、簡化或改寫。

請固定使用以下標準名稱：

- double_bottom → 雙重底
- double_top → 雙重頂
- head_and_shoulders_bottom → 頭肩底
- head_and_shoulders_top → 頭肩頂
- ascending_triangle → 上升三角形
- descending_triangle → 下降三角形
- symmetrical_triangle → 對稱三角形
- box → 箱型整理
- flag → 旗形

【禁止】
- 雙重底 → 雙底 / W底
- 上升三角形 → 收斂三角 / 三角整理
- 箱型整理 → 其他名稱
- 旗形 → 小旗 / 整理旗

【重要】
- 型態名稱只能有一種寫法
- 不可使用同義詞
- 不可口語化簡寫
- 這是為了確保輸出一致性與可驗證性

--------------------------------------------------

【型態必須直接使用 structure（強制）】

最終「型態」欄位：

1. 主型態名稱
   → 必須直接使用 structure 中的 structure_type
   → 不可重新判斷或改寫

2. 型態日期
   → 必須優先使用 structure_range
   → 不可自行改寫為其他時間區間
   → 不可延伸、縮短或模糊化

【禁止】
- 改寫型態（例如：上升三角形 → 高檔整理）
- 改寫時間（例如：1月～3月 → 11月～2月）
- 自行重新判斷型態區間

【允許】
- 將 structure_range 轉成自然語句
  例如：
  - 1月13日至3月20日
  - 2月下旬至3月20日
  - 2025年11月21日至2026年3月20日

【重要】
structure 是最終型態的唯一來源，不是參考來源。

--------------------------------------------------

【structure 中的 none 不可外洩（強制）】

若 structure_type = none 或 structure_range = none：

- 不可在最終輸出中出現：
  - none
  - none / none
  - （none）
  - 任何把 none 直接當成內容的寫法

【規則】
1. 若 structure_type = none：
   - 「型態」欄位必須改寫為自然語言
   - 例如：
     - 高檔整理
     - 區間盤整
     - 狹幅盤整
     - 多頭整理
     - 偏空震盪

2. 若 structure_range = none 或 空字串：
   - 不可補日期
   - 不可寫任何佔位符
   - 直接描述目前整理狀態即可

【正確寫法】
- 目前走勢屬於區間盤整，約在33500點到35000點之間來回整理。
- 現階段為高檔整理，上方約35000點壓力仍在，下方約33500點附近有支撐。

【禁止寫法】
- 區間盤整（none / none）
- 高檔整理（none）
- 型態：none

--------------------------------------------------

【單一主型態（強制）】

「型態」欄位只能有一個主型態。

選擇原則：
1. 最近且仍有效
2. 已成立優先
3. 層級較大的型態優先

【重要】
- 較短線、較次階的小型態只能當補充
- 不可並列兩個型態
- 不可先寫主型態，再完整展開第二個型態

--------------------------------------------------

【型態不可輸出 none（強制）】

若沒有明確技術型態：

- 不可輸出 none
- 不可寫「沒有型態」
- 不可直接暴露內部判斷值

此時必須改寫為使用者可理解的自然語言，例如：
- 高檔整理
- 區間盤整
- 狹幅盤整
- 多頭整理
- 偏空震盪

【重要】
- 即使沒有明確型態，若屬箱型震盪 / 區間盤整 / 狹幅盤整
  仍要清楚交代支撐與壓力位置（價格或點位）
- 該提數字時仍要提數字，不能因為不是典型型態就省略關鍵價位

--------------------------------------------------

【型態描述簡潔化（強制）】

「型態」欄位必須簡潔，避免過度解釋。

【長度限制】
- 以 1～2 句為主
- 不可寫成長段分析

【內容優先順序】
只保留：
1. 型態名稱
2. 型態時間（用於回圖）
3. 當前位置或狀態

【不需要】
- 詳細形成過程
- 結構推導
- 重複描述高低點邏輯
- 與其他欄位重複內容

--------------------------------------------------

【語氣限制（強制）】

禁止使用：
- 主軸
- 結構如下
- 型態如下
- 此型態代表
- 完成度：
- 起訖日期：
- 當前位置：
- 目標價：
- 使用括號列點式輸出

改用自然說法：
- 目前走勢屬於...
- 可以看到...
- 這段走勢形成...

--------------------------------------------------

【型態日期（允許且建議）】

型態可以提供明確起訖日期，用來幫助使用者回到圖表確認型態。

【原則】
- 日期要融入句子中，不可用欄位語氣
- 可使用明確日期，例如：
  - 1月8日至3月20日
  - 2月下旬至3月20日
  - 2025年11月21日至2026年3月20日
- 不可寫成：
  - 起訖日期：...
  - （1月8日～3月20日）

【目的】
- 讓使用者可以在看盤軟體上自行複查
- 增加型態的可驗證性

--------------------------------------------------

【型態欄位建議句型】

請優先使用以下結構：

「型態 + 日期 + 現況」

例如：
- 目前走勢屬於雙重底，約在1月8日至3月20日形成，近期在頸線附近震盪整理。
- 這段走勢為箱型整理，約落在33000點到35000點之間，目前仍在區間內來回。
- 目前走勢屬於高檔整理，上方約35000點壓力仍在，下方約33500點附近有支撐。

【壓縮原則】
若句子過長：
- 優先刪除過程描述
- 保留結果 + 位置

--------------------------------------------------

【最終輸出必須整合為單一觀點（強制）】

long / mid / short 只用於內部分析，不可直接照搬到最終輸出。

所有欄位都必須整合長、中、短週期後，寫成單一結論。

【禁止寫法】
- 長線仍屬...
- 中線則是...
- 短線來看...
- 從長中短週期來看...
- 長期...；中期...；短期...

【正確原則】
- 長週期提供方向
- 中週期提供結構
- 短週期提供節奏
- 最終輸出只能呈現整合後的結果

--------------------------------------------------

【K線圖欄位整合規則（強制）】

「K線圖」欄位必須是整體走勢的整合描述，不可拆成長線 / 中線 / 短線分別說明。

應直接回答：
- 現在整體走勢處於哪個階段
- 當前結構偏多、偏空或整理
- 價格目前卡在哪個關鍵位置
- 接下來最可能的節奏是什麼

--------------------------------------------------

【用語在地化與語氣控制（強制）】

必須使用台灣技術分析常用語。

【禁止用詞】
- 箱體
- 箱體整理
- 箱體震盪
- trading range

【用詞選擇邏輯】
- 波動很小 → 狹幅整理
- 小震盪 → 整理 / 盤整
- 有上下界 → 區間整理 / 區間震盪
- 高位 → 高檔整理
- 低位 → 低檔整理

【型態轉換】
若 structure_type = box：
→ 可依語境使用：
  - 箱型整理
  - 區間整理
  - 區間震盪
  - 高檔箱型整理
  - 低檔箱型整理
→ 但若是主型態名稱，仍優先固定寫為「箱型整理」
→ 禁止使用「箱體」

--------------------------------------------------

【移動平均線判讀規則（強制）】

均線分析必須明確指出使用的均線，不可只寫「均線」或「均線帶」。

【均線定義】

短期均線：
- 5日線
- 10日線

中期均線：
- 20日線
- 60日線

--------------------------------------------------

【均線趨勢判斷（強制）】

請依以下規則判斷：

- 5日線 + 10日線 同時上揚 → 短線偏多
- 5日線 + 10日線 同時下彎 → 短線偏空
- 其他 → 短線盤整

- 20日線 + 60日線 同時上揚 → 中期偏多
- 20日線 + 60日線 同時下彎 → 中期偏空
- 其他 → 中期盤整

--------------------------------------------------

【均線輸出規則（強制）】

必須明確寫出：
- 哪些均線提供支撐或壓力
- 不可只寫「均線支撐」

【正確寫法】
- 5日線與10日線上揚，短線仍有支撐
- 價格回測20日線附近有承接
- 60日線仍下彎，形成上方壓力

【禁止寫法】
- 均線支撐
- 均線帶
- 均線附近（未說明是哪條）

--------------------------------------------------

【均線價位對應（重要）】

當提到支撐或壓力時：
- 必須對應具體價格區間
- 並說明該價位對應哪條均線

例如：
- 約27.8～28.2元為10日線與20日線重疊區
- 約27元附近為60日線支撐

【重要】
均線的目的：
👉 提供可驗證的支撐 / 壓力依據
不可寫成模糊描述

--------------------------------------------------

【技術指標用語白名單（強制遵守）】

【KD指標】

允許：
- K線
- D線
- KD雙線
- 超買區 / 超賣區
- 高檔 / 低檔
- 鈍化
- 黃金交叉 / 死亡交叉
- 回升 / 回落

禁止：
- 偏強區 / 偏弱區
- 強勢區 / 弱勢區
- 動能區

【KD主詞規則（強制）】
- 不要只寫「KD...」
- 必須明確：
  - K線 / D線 / KD雙線

【KD圖形 → 技術語言轉換（強制）】
KD圖中的綠線僅為輔助判讀，不可出現在輸出中。

【禁止描述】
- 上方綠線
- 下方綠線
- 兩條綠線之間
- 接近綠線

【必須轉換】
- 接近上方綠線 → 接近超買區 / 接近高檔
- 高於上方綠線 → 進入超買區
- 接近下方綠線 → 接近超賣區 / 接近低檔
- 低於下方綠線 → 進入超賣區

【補充規則】
- 若無法確認是否進入超買 / 超賣：
  → 使用「接近高檔 / 接近低檔」
- 若描述交叉或同步變化：
  → 優先使用「KD雙線」
- 若描述快慢差異：
  → 明確區分 K線 與 D線

--------------------------------------------------

【MACD指標】

允許：
- DIF線
- MACD線
- OSC柱體
- 零軸
- 多方 / 空方
- 動能增強 / 減弱
- 柱體擴大 / 縮小
- 背離

禁止：
- 訊號線
- macd / macdsignal / macdhist

--------------------------------------------------

【成交量（重要）】

優先使用：
- 價漲量增
- 價跌量縮
- 量縮整理

輔助：
- 量能放大 / 縮小
- 換手
- 價量背離

【成交量時間尺度規則（強制）】
成交量只允許使用短週期（60根）作為內部判讀依據。

【重要】
- 「60根」是內部分析規格，不可直接寫進最終輸出
- 不可寫出：
  - 近60根量能
  - 最近60根成交量
  - 從60根來看
- 最終輸出只可直接描述量價現象與量能狀態
- 應寫成自然語言，例如：
  - 近期量能偏向量縮整理
  - 反彈時未見明顯放量
  - 價格靠近壓力區時追價量能不足

【價量背離判斷（強制）】
必須主動檢查是否背離。

若背離：
→ 必須寫在「成交量」與「整體評價」
→ 並提高權重（視為風險）

--------------------------------------------------

【數據使用規範（強制）】

可以用數據判斷，但：

❌ 不可提及資料來源
❌ 不可用欄位名稱
❌ 不可像資料表

【關鍵數字可輸出，但必須加單位】

你可以輸出以下關鍵數字，但必須使用 tool 提供的單位：
- 支撐 / 壓力
- 買點 / 停損 / 目標價
- 均線價位
- 成交量
- 少量必要的指標數值

【單位規則（強制）】
請依 instrument 中的單位資訊輸出：
- 價格 / 指數點位 → 使用 price_unit
- 成交量 → 使用 volume_unit

例如：
- 若 price_unit = 點，應寫：35000點、33500點附近
- 若 price_unit = 元，應寫：120元、118元附近
- 若 volume_unit = 億元，應寫：成交量放大至約3000億元
- 若 volume_unit = 張，應寫：成交量約5萬張

【禁止】
- 只寫數字不帶單位
- 同一份輸出中單位前後不一致
- 使用原始資料格式，例如：
  - K=84.91、D=77.51、Volume=123456
  - macd=1.23

【指標數值規則】
- 指標數值只有在真的有助於判斷時才可少量引用
- 即使引用，也要改寫成自然技術分析語言
- 不可像資料表逐項列值

--------------------------------------------------

【量化語氣自動重寫（強制）】

若出現以下情況，必須重寫：
- 原始欄位名稱（macd / slowk / slowd 等）
- 資料表式數字堆疊
- 工程語言或資料來源語言
- 過度量化、像報表而不像解盤

注意：
- 關鍵數字可以保留
- 但必須加上正確單位
- 並改寫成自然的技術分析語言

【範例】
❌ K=84.91、D=77.51
✅ K線約在85附近，已進入超買區，D線仍維持高檔

❌ macd=1.23
✅ DIF線仍在零軸上方，多方動能維持

❌ 35000
✅ 35000點附近壓力仍在

❌ 50000
✅ 成交量約5萬張

--------------------------------------------------

【K棒規則】

K棒只代表短期（3~5天）的節奏判讀。

【重要】
- 「3~5天」是內部分析範圍，不可直接出現在最終輸出
- 不可寫出：
  - 近3~5天
  - 最近3~5日
  - 短期3~5天K棒
- 最終輸出只能直接描述看到的K棒節奏與市場反應
- 應寫成自然語言，例如：
  - 近期K棒以小紅小黑交錯為主
  - 短線拉回後仍有承接
  - 連續收黑顯示追價意願降溫

--------------------------------------------------

【日期格式規範（強制）】

日期必須使用：
👉 M月D日（不可補零）

❌ 03/17
✅ 3月17日

--------------------------------------------------

【K棒日期提及規則（強制）】

只有在以下情況才可提日期：
- 關鍵轉折（止跌 / 轉弱 / 突破）
- 關鍵K棒（長紅 / 長黑 / 爆量）
- 節奏需要說明時

不可：
- 每天都報
- 逐日列舉

【原則】
日期是標記重點，不是逐日記錄

--------------------------------------------------

【標的代號不可出現在輸出文字中（強制）】

最終輸出的各欄位內容不可直接出現：
- 股票代號
- 指數代號
- ETF代號
- 英文代碼
- 標的名稱作為句首主詞

【禁止寫法】
- TAIEX目前屬...
- 2330目前...
- 0050目前...
- 本標的目前...

【正確寫法】
- 目前走勢屬於...
- 現階段結構偏向...
- 整體來看屬於...
- 當前盤勢仍是...

--------------------------------------------------

【內部規格不可外顯（強制）】

以下內容屬於模型內部分析依據，只能用來幫助判讀，不可直接出現在最終輸出文字中：

- 3~5天
- 60根
- 長週期 / 中週期 / 短週期
- 240 / 120 / 60
- 結構判斷流程
- 指標判讀門檻
- 任何「我是根據某個固定視窗分析」的說法
- Phase 1 / Phase 2
- structure_type / long_view / mid_view / short_view / structure_range

【禁止寫法】
- 近3~5天K棒...
- 近60根量能...
- 從短週期來看...
- 從120根結構觀察...
- 依據Phase 1結構...

【輸出原則】
最終輸出只能呈現：
- 圖上看到的現象
- 技術分析語言下的判讀結果
- 使用者可直接閱讀的自然語句

不可呈現：
- 分析流程
- 判斷規格
- 規則來源
- 視窗長度
- 內部工作方式

--------------------------------------------------

【輸出 JSON】

{{
  "K棒": "...",
  "K線圖": "...",
  "成交量": "...",
  "型態": "...",
  "移動平均線": "...",
  "MACD指標": "...",
  "KD指標": "...",
  "整體評價": "..."
}}
""".strip()

    if price_data:
        prompt += "\n\n補充數據：\n" + json.dumps(price_data, ensure_ascii=False)

    return prompt


# =========================
# AGENT
# =========================

class StockAnalysisAgent:
    def __init__(self, model: str = "gpt-5.2", sqlite_db_path = 'data/stock.db', debug = False):
        # API Key不明寫，從環境變數中取得
        load_dotenv(find_dotenv())
        api_key = os.environ.get('OPENAI_API_KEY_TOKEN')
        self.client = OpenAI(api_key=api_key)
        self.model = model
        # 我的內部屬性
        ### 設定除錯旗標 ###
        self._debug = debug
        ### 設定股票代號 ###
        self._stock_id = None
        # 我的初始化程序
        ### 連線資料庫 ###
        if os.path.isfile(sqlite_db_path):
            self._conn = sqlite3.connect( sqlite_db_path)
        else :
            raise ValueError('The database file does not exist.')
        ### 從資料庫中載入「台股總覽 TaiwanStockInfo」 ###
        try :
            self._df_stock_info = pd.read_sql("SELECT * FROM StockInfo", self._conn)
        except Exception as e:
            raise ValueError('An error occurred while loading TaiwanStockInfo.')

        self.lookbacks = {
            "long": 240,
            "mid": 120,
            "short": 60,
        }

    # 我的內部方法
    ### 列印除錯訊息之內部方法 ###
    def _debug_print( self, msg) :
        if self._debug is True :
            print("ＤＥＢＵＧ ： {}".format(msg))
    ### 重置內部屬性之內部方法 ###
    def _reset_internal_attribute( self) :
        self._daily_price_df        = None
        self._sma_df                = None
        self._kd_df                 = None
        self._macd_df               = None
        self._instrument_type       = None
        self._price_unit            = None
        self._volume_unit           = None
        ### 從資料庫中載入日Ｋ資料之內部方法 ###
    def _loading_price_data( self, stock_id) :
        # 將載入的「台股總覽 TaiwanStockInfo」進行格式轉換
        df_stock_info = self._df_stock_info.set_index(self._df_stock_info['StockID'],inplace=False)
        df_stock_info = df_stock_info.drop(columns=['StockID'])
        
        # 判斷股票代碼(stock_id)是否存在於「台股總覽 TaiwanStockInfo」中
        if stock_id in df_stock_info.index:
            
            # 取得該股票代碼的產業分類
            individual_stock_info = df_stock_info.loc[stock_id]
            industry_category     = individual_stock_info['IndustryCategory']
            
            # 設定開始日期與結束日期
            current_date       = datetime.datetime.today()
            daily_end_date     = current_date.strftime('%Y-%m-%d')
            daily_start_date,_ = get_monday_to_sunday((current_date - datetime.timedelta(days=730)).strftime('%Y-%m-%d'))
            
            # 讀取日Ｋ價格資料
            sql_cmd = "SELECT * FROM DailyPrice WHERE StockID='{}' AND (Date BETWEEN '{}' AND '{}') ORDER BY Date".format(stock_id,daily_start_date,daily_end_date)
            try :
                daily_price_df = pd.read_sql( sql_cmd, self._conn)
            except Exception as e:
                self._debug_print('讀取日Ｋ資料錯誤，錯誤訊息＝ {}'.format(str(e)))
                return False
            # 調整停牌或未交易之日Ｋ價格資料
            correcting_zero_price_issue( daily_price_df, debug = self._debug)
            # 格式轉換：日期格式、成交量(成交值)
            daily_price_df           = daily_price_df.drop(columns=['SerialNo','StockID'])
            daily_price_df['Date']   = daily_price_df['Date'].astype('datetime64[ns]')
            daily_price_df.set_index(daily_price_df['Date'],inplace=True)
            daily_price_df           = daily_price_df.drop(columns=['Date'])
            if industry_category == 'Index' or industry_category == '大盤' :
                daily_price_df           = daily_price_df.drop(columns=['Volume'])
                daily_price_df           = daily_price_df.rename(columns={'Value':'Volume'})
                daily_price_df['Volume'] = daily_price_df['Volume'].div(100000000.00)
                daily_price_df['Volume'] = daily_price_df['Volume'].round(2)
            else :
                daily_price_df           = daily_price_df.drop(columns=['Value'])
                daily_price_df['Volume'] = daily_price_df['Volume'].div(1000)
                daily_price_df['Volume'] = daily_price_df['Volume'].round()
                daily_price_df['Volume'] = daily_price_df['Volume'].astype('int64')
            self._daily_price_df         = daily_price_df
                
            if industry_category == 'Index' or industry_category == '大盤' :
                self._instrument_type       = 'index'
                self._price_unit            = '點'
                self._volume_unit           = '億元'
            else :
                self._instrument_type       = 'stock'
                self._price_unit            = '元'
                self._volume_unit           = '張'
                
            return True
        return False
    ### 使用talib程式庫計算技術指標之內部方法 ###
    def _technical_indicators( self) :
        # 日Ｋ價格資料轉換為talib格式
        daily_price_df_talib          = self._daily_price_df.copy()
        daily_price_df_talib.columns  = [ i.lower() for i in daily_price_df_talib.columns]
        
        # 計算移動平均線
        talib_sma5        = SMA( daily_price_df_talib, timeperiod=5)
        talib_sma10       = SMA( daily_price_df_talib, timeperiod=10)
        talib_sma20       = SMA( daily_price_df_talib, timeperiod=20)
        talib_sma60       = SMA( daily_price_df_talib, timeperiod=60)
        talib_sma120      = SMA( daily_price_df_talib, timeperiod=120)
        talib_sma240      = SMA( daily_price_df_talib, timeperiod=240)
        # 設定名稱
        talib_sma5.name   = 'SMA5'
        talib_sma10.name  = 'SMA10'
        talib_sma20.name  = 'SMA20'
        talib_sma60.name  = 'SMA60'
        talib_sma120.name = 'SMA120'
        talib_sma240.name = 'SMA240'
        # 合併各條均線
        talib_sma_df      = pd.concat([talib_sma5, talib_sma10, talib_sma20, talib_sma60, talib_sma120, talib_sma240], axis=1)
        # 取小數點後兩位
        self._sma_df      = talib_sma_df.round(2)
        
        # 計算ＫＤ指標
        talib_daily_kd = STOCH( daily_price_df_talib, fastk_period=9, slowk_period=3, slowd_period=3)
        # 取小數點後兩位
        self._kd_df    = talib_daily_kd.round(2)
        
        # 計算ＭＡＣＤ指標
        talib_daily_macd = MACD( daily_price_df_talib, fastperiod=12, slowperiod=26, signalperiod=9)
        # 取小數點後兩位
        self._macd_df    = talib_daily_macd.round(2)
    ### 載入價量資料並計算技術指標 ###
    def _loading_price_data_and_technical_indicators( self, stock_id) :
        if self._stock_id is None or self._stock_id != stock_id :
            self._debug_print('設置或重置內部屬性')
            self._reset_internal_attribute()
            self._debug_print('載入價量資料,股票代碼 ＝ {}'.format(stock_id))
            if self._loading_price_data( stock_id) is True :
                self._debug_print('計算技術指標')
                self._technical_indicators()
                self._stock_id = stock_id
            else :
                raise ValueError("讀取價量資料錯誤！")

    # ===== TODO =====

    def classify_instrument(self, stock_id: str) -> Dict[str, str]:
        
        self._loading_price_data_and_technical_indicators( stock_id)
        
        return {"instrument_type":self._instrument_type , "price_unit":self._price_unit , "volume_unit": self._volume_unit}

    def get_analysis_dates(self, prev_days: int) -> Tuple[str, str]:

        if  self._daily_price_df is not None :
            end_date   = self._daily_price_df.iloc[-1].name.strftime('%Y-%m-%d')
            start_date = self._daily_price_df.iloc[-1-prev_days].name.strftime('%Y-%m-%d')
            self._debug_print('開始日期 ＝ {} （輸入參數 ＝ {} ）， 結束日期 ＝ {} '.format(start_date,prev_days,end_date))
        else :
            raise ValueError("沒有價量資料！")
        
        return (start_date,end_date)


    def get_chart(self, stock_id: str, start_date: str, end_date: str) -> str:
        
        self._loading_price_data_and_technical_indicators( stock_id)
        
        range_prices = self._daily_price_df[start_date:end_date]
        range_sma    = self._sma_df[start_date:end_date]
        range_kd     = self._kd_df[start_date:end_date]
        range_macd   = self._macd_df[start_date:end_date]
        
        # 設定K線格式
        mc = mpf.make_marketcolors(up='xkcd:light red', down='xkcd:almost black', inherit=True)
        s  = mpf.make_mpf_style(base_mpf_style='yahoo', marketcolors=mc)
        
        # ＫＤ指標：超買線
        kd_overbought_line = [80] * range_prices.shape[0]
        # ＫＤ指標：賣超線
        kd_oversold_line   = [20] * range_prices.shape[0]

        # 設定技術指標
        added_plots = [
            mpf.make_addplot(range_sma['SMA5'],width=0.8,color='xkcd:red brown'),
            mpf.make_addplot(range_sma['SMA10'],width=0.8,color='xkcd:dark sky blue'),
            mpf.make_addplot(range_sma['SMA20'],width=0.8,color='xkcd:violet'),
            mpf.make_addplot(range_sma['SMA60'],width=0.8,color='xkcd:orange'),
            mpf.make_addplot(range_kd['slowk'],width=0.8,panel=2,secondary_y=False,color='xkcd:red',ylabel='KD'),
            mpf.make_addplot(range_kd['slowd'],width=0.8,panel=2,secondary_y=False,color='xkcd:blue'),
            mpf.make_addplot(kd_overbought_line,width=0.8,panel=2,secondary_y=False,linestyle='--',color='xkcd:green'),
            mpf.make_addplot(kd_oversold_line,width=0.8,panel=2,secondary_y=False,linestyle='--',color='xkcd:green'),
            mpf.make_addplot(range_macd['macdhist'],type='bar',panel=3,secondary_y=False,color='xkcd:grey',ylabel='MACD'),
            mpf.make_addplot(range_macd['macd'],width=0.8,panel=3,secondary_y=False,color='xkcd:red'),
            mpf.make_addplot(range_macd['macdsignal'],width=0.8,panel=3,secondary_y=False,color='xkcd:blue')
        ]

        # 設定Ｋ線圖X軸座標值
        ticks        = []
        tlabs        = []
        label_count  = 0
        total_k_line = range_prices.shape[0]
        for idx in range(total_k_line) :
            timestamp = range_prices.iloc[idx].name
            if idx == 0 :
                ticks.append(idx)
                tlabs.append(timestamp.strftime('%Y-%m-%d'))
                label_count = 0
            elif (idx+1) == total_k_line:
                if label_count < 5 :
                    # 移除上一筆資料
                    ticks.pop()
                    tlabs.pop()
                ticks.append(idx)
                tlabs.append(timestamp.strftime('%Y-%m-%d'))
            elif label_count > 10 :
                ticks.append(idx)
                tlabs.append(timestamp.strftime('%Y-%m-%d'))
                label_count = 0
            label_count += 1
        
        # 繪製Ｋ線圖
        tmp = io.BytesIO()
        kwargs = dict(type='candle', style=s, figratio=(16,9), volume=True, addplot=added_plots, main_panel=0, volume_panel=1, panel_ratios=(5, 1, 2, 2), num_panels=4, datetime_format='%Y-%m-%d', returnfig=True, savefig=dict(fname=tmp))
        fig, axlist = mpf.plot(range_prices,**kwargs)
        axlist[-2].set_xticks(ticks,labels=tlabs,ha='right')
        tmp = None
        
        # 保存Ｋ線圖
        img = io.BytesIO()
        fig.savefig(img,format='png')

        # 刪除空白
        image    = Image.open(img).convert("RGB")
        image    = crop_borders(image, crop_color=(255, 255, 255))
        crop_img = io.BytesIO()
        image.save(crop_img,format='PNG')
        
        # 將影像檔(PNG)編碼為Base64
        base64_image = base64.b64encode(crop_img.getvalue()).decode()
        
        return base64_image

    def get_price_data(self, stock_id: str, start_date: str, end_date: str) -> Dict[str, Any]:
        
        self._loading_price_data_and_technical_indicators( stock_id)
        
        range_prices     = self._daily_price_df[start_date:end_date]
        range_sma        = self._sma_df[start_date:end_date]
        range_kd         = self._kd_df[start_date:end_date]
        range_macd       = self._macd_df[start_date:end_date]
        
        ret_records_list = []
        
        for idx in range(len(range_prices)) :
            record = {
                "date": range_prices.iloc[idx].name.strftime('%Y-%m-%d'),
                "open": range_prices.iloc[idx]['Open'],
                "high": range_prices.iloc[idx]['High'],
                "low": range_prices.iloc[idx]['Low'],
                "close": range_prices.iloc[idx]['Close'],
                "volume": range_prices.iloc[idx]['Volume'],
                "sma5": range_sma.iloc[idx]['SMA5'],
                "sma10": range_sma.iloc[idx]['SMA10'],
                "sma20": range_sma.iloc[idx]['SMA20'],
                "sma60": range_sma.iloc[idx]['SMA60'],
                "macd": range_macd.iloc[idx]['macd'],
                "macdsignal": range_macd.iloc[idx]['macdsignal'],
                "macdhist": range_macd.iloc[idx]['macdhist'],
                "slowk": range_kd.iloc[idx]['slowk'],
                "slowd": range_kd.iloc[idx]['slowd'] 
            }
            ret_records_list.append(record)
        
        summary = {'latest_close' : range_prices.iloc[-1]['Close']}
            
        return {"records":ret_records_list,"summary":summary}

    # ===== internal =====

    def _image_url(self, b64: str) -> str:
        return f"data:image/png;base64,{b64}"

    def _safe_json(self, text: str) -> dict:
        text = text.strip()
        if text.startswith("```"):
            lines = text.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].startswith("```"):
                lines = lines[:-1]
            text = "\n".join(lines).strip()
            if text.lower().startswith("json"):
                text = text[4:].strip()
        return json.loads(text)

    def _call(self, system: str, user: str, images: dict) -> str:
        content = [{"type": "input_text", "text": user}]
        for k in ["long", "mid", "short"]:
            content.append({"type": "input_image", "image_url": images[k]})

        r = self.client.responses.create(
            model=self.model,
            input=[
                {"role": "system", "content": system},
                {"role": "user", "content": content}
            ]
        )
        return r.output_text

    def _get_charts(self, stock_id: str) -> dict:
        result = {}
        for k, days in self.lookbacks.items():
            s, e = self.get_analysis_dates(days)
            b64 = self.get_chart(stock_id, s, e)
            result[k] = self._image_url(b64)
        return result

    # ===== Phase 1 =====

    def extract_structure(self, stock_id: str) -> dict:
        instrument = self.classify_instrument(stock_id)
        charts = self._get_charts(stock_id)

        raw = self._call(
            build_system_prompt(instrument),
            build_phase1_prompt(stock_id),
            charts
        )
        return self._safe_json(raw)

    # ===== Phase 2 =====

    def generate_analysis(self, stock_id: str, structure: dict) -> dict:
        instrument = self.classify_instrument(stock_id)
        charts = self._get_charts(stock_id)

        price_data = None
        if structure.get("need_price_data") == "YES":
            s, e = self.get_analysis_dates(120)
            price_data = self.get_price_data(stock_id, s, e)

        raw = self._call(
            build_system_prompt(instrument),
            build_phase2_prompt(stock_id, structure, instrument, price_data),
            charts
        )
        return self._safe_json(raw)

    # ===== API =====

    def run(self, stock_id: str, debug: bool = False):
        structure = self.extract_structure(stock_id)
        analysis = self.generate_analysis(stock_id, structure)

        if debug:
            return {
                "結構": structure,
                "分析": analysis
            }
        return analysis

In [4]:
agent = StockAnalysisAgent(debug=True)

In [5]:
# 測試：加權指數
result = agent.run('TAIEX',debug=True)
print('除錯資訊 ＝  {}'.format(result))
display_result(result['分析'])

ＤＥＢＵＧ ： 設置或重置內部屬性
ＤＥＢＵＧ ： 載入價量資料,股票代碼 ＝ TAIEX
ＤＥＢＵＧ ： 計算技術指標
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
除錯資訊 ＝  {'結構': {'stock_id': 'TAIEX', 'long_view': {'structure_type': 'none', 'structure_range': '2025-03-25 ~ 2026-03-20', 'note': '近一年走勢以「高點墊高、回檔不破前波關鍵低點」為主，未見最近且仍主導當前價格的頭肩頂/雙重頂等明確反轉型態；近期屬創高後回落、在短中期均線附近整理。支撐以長均線與前波回檔低點區為主，壓力在前高區（約35000點附近）。KD自低檔回升；MACD的DIF線與MACD線在高檔回落後趨於收斂，OSC柱體由正轉弱，顯示動能降溫但尚未形成清楚反轉結構。'}, 'mid_view': {'structure_type': 'none', 'structure_range': '2025-09-15 ~ 2026-03-20', 'note': '中週期先急漲創高後快速修正，隨後在約33000~34500點之間反覆拉鋸，較像「漲後整理」而非已完成的反轉型態（未見明確雙頂頸線跌破後的續跌結構）。目前價格靠近短中期均線糾結處，屬整理末端偏回穩。支撐看前低反彈區（約33000點附近）與中期均線；壓力仍在前高區（約35000點附近）。KD回到中高檔；MACD的DIF線仍在回落段、與MACD線貼近，OSC柱體縮

技術分析工具,解盤內容
K棒,近期K棒以小紅小黑交錯為主，創高後回落但下方回測仍見承接，顯示追價降溫、但也尚未出現連續轉弱的走法。
K線圖,整體仍屬漲多後的高檔整理格局，指數在34000點附近反覆拉鋸；上方靠近34500點一帶容易遇壓，下方回到33500～33700點附近有支撐，屬區間內等待表態的節奏，需靠放量帶方向。
成交量,量能大致中性偏整理，反彈靠近壓力區時未見明顯追價放量，盤勢更像是在區間內換手而非趨勢續攻；目前未見明顯價量背離訊號。
型態,這段走勢為箱型整理，約在3月10日至3月20日形成，區間大致落在33500～34500點，目前仍在箱內震盪。
移動平均線,5日線與10日線彼此貼近、走勢偏平，短線以盤整看待；20日線與60日線在34000點附近交疊，成為目前主要的拉鋸帶。上方壓力先看34500點附近（短期均線密集區與前波反壓），若能站穩才有機會再挑戰35000點附近前高；下方支撐看33700點附近（20日線附近）與33500點附近（箱底區），再下來則是60日線附近約33000點一帶。
MACD指標,DIF線在零軸附近走平偏弱，與MACD線貼近收斂；OSC柱體偏小且由正轉弱，顯示多方動能降溫、目前以整理等待方向為主，需看到柱體再度擴大才較有續攻力道。
KD指標,K線、D線自低檔回升到中高檔，但尚未出現明顯鈍化；在區間上緣時容易反覆，短線較像是整理中的回升，而非一路推升的強趨勢段。
整體評價,偏多架構下的高檔箱型整理：操作上以區間應對較合適，34500點附近若放量突破，才有利延伸到35000點附近再測前高；若跌破33500～33700點支撐，整理可能轉為較深的回檔，風險控管以箱底失守為優先觀察。


In [6]:
# 測試：宏碁(2353)
result = agent.run('2353',debug=True)
print('除錯資訊 ＝  {}'.format(result))
display_result(result['分析'])

ＤＥＢＵＧ ： 設置或重置內部屬性
ＤＥＢＵＧ ： 載入價量資料,股票代碼 ＝ 2353
ＤＥＢＵＧ ： 計算技術指標
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
除錯資訊 ＝  {'結構': {'stock_id': '2353', 'long_view': {'structure_type': 'double_bottom', 'structure_range': '2025-11-25 ~ 2026-02-10', 'note': '長週期先看底型：11月底/12月初一波下殺後，1月下旬再回測低點區（約25元附近）未再破前低，之後價格重新走高並站回短中期均線之上；近期彈升逼近/突破前波整理高點區（約28.5~29元）。量能在突破段有放大跡象。DIF線上穿MACD線後維持在零軸附近偏上，OSC柱體翻正擴大，偏向底型後的回升延續。上方壓力看29~30元與長均線下彎區；下方支撐看27元附近（均線糾結處）與25~25.5元（雙底低點）。'}, 'mid_view': {'structure_type': 'double_bottom', 'structure_range': '2025-12-23 ~ 2026-02-10', 'note': '中週期與長週期一致，仍以雙重底為主軸：12/下旬落底後，2/上旬再測未破（約25.5元附近），後續沿均線上推並在3月突破整理區上緣（約28~28.5元），屬底型完成後的推進段。DIF線維持在MACD線之上且靠近零軸上方，OSC柱體為正並增強，偏多。壓力落在29~30元

技術分析工具,解盤內容
K棒,近期先拉出一段紅K推升到29元附近後，出現高檔黑紅交錯、上影線較明顯的震盪，追價力道有降溫，但回吐幅度不大，價格仍多半守在28元附近。
K線圖,目前走勢屬於雙重底，約在2025年11月25日至2026年2月10日形成，之後沿著墊高走勢推進，近期來到前高壓力帶29～30元附近轉為高檔震盪；只要回檔能守住28元一帶，仍偏向以整理代替拉回，續攻機會較高。
成交量,突破上來的段落可見量能放大，推升過程屬於價漲量增；但到29元附近後量能沒有再明顯擴大，呈現量縮震盪，屬於壓力區常見的整理節奏，暫未看到明顯價量背離的轉弱訊號。
型態,目前走勢屬於雙重底，約在2025年11月25日至2026年2月10日形成，近期來到29～30元壓力區附近震盪整理。
移動平均線,5日線與10日線仍在上揚，短線續強的架構尚在，但拉回時會先看約27.8～28.2元（10日線與20日線附近）能否守住；20日線走平偏上、60日線仍偏下彎，代表中期剛轉穩但上方約29～30元仍會受到60日線與前高壓力牽制，需用時間消化。
MACD指標,DIF線維持在MACD線之上並靠近零軸上方，OSC柱體維持正值但擴大速度放慢，顯示多方動能仍在、但進入壓力區後動能轉為收斂，較符合高檔整理而非反轉走弱。
KD指標,K線與D線位在高檔後出現回落，屬於高檔鈍化後的降溫；若後續整理期間K線能再度上彎並帶動KD雙線黃金交叉，較有利再挑戰29～30元，反之跌破28元則整理時間可能拉長。
整體評價,整體仍是雙重底後的回升延續格局，現在卡在29～30元壓力帶，較可能先高檔震盪消化賣壓。操作上以「守住28元續抱、跌破28元偏保守」作為強弱分界；上方先看29～29.5元，能帶量站穩才有機會續看30元附近，反之回測支撐則依序看27.2～27.5元與25.5～26元。


In [7]:
# 測試：大成(1210)
result = agent.run('1210',debug=True)
print('除錯資訊 ＝  {}'.format(result))
display_result(result['分析'])

ＤＥＢＵＧ ： 設置或重置內部屬性
ＤＥＢＵＧ ： 載入價量資料,股票代碼 ＝ 1210
ＤＥＢＵＧ ： 計算技術指標
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-03-25 （輸入參數 ＝ 240 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-09-15 （輸入參數 ＝ 120 ）， 結束日期 ＝ 2026-03-20 
ＤＥＢＵＧ ： 開始日期 ＝ 2025-12-12 （輸入參數 ＝ 60 ）， 結束日期 ＝ 2026-03-20 
除錯資訊 ＝  {'結構': {'stock_id': '1210', 'long_view': {'structure_type': 'double_bottom', 'structure_range': '2025-10-20 ~ 2026-03-20', 'note': '長週期先看型態：2025/10下旬出現一個明顯低點後反彈，之後回測到2026/02~03附近形成第二個相近低點且未再破前低，近期價格重新走高並站回中短期均線之上，底部型態仍對當前價格具主導。量能在突破走高階段有放大跡象。DIF線上彎並貼近/上穿MACD線，OSC柱體翻正擴大，屬底部完成後的推進段。關鍵壓力落在55附近（前波整理上緣/近期高點帶），支撐先看52附近（整理區與均線帶）。'}, 'mid_view': {'structure_type': 'double_bottom', 'structure_range': '2025-12-20 ~ 2026-03-20', 'note': '中週期需與長週期一致，主軸維持雙重底：第一腳低點約落在2025/12下旬~2026/01初，反彈後於2026/02下旬~03上旬再回測形成第二腳低點（低點相近、未破前低），隨後拉升突破整理帶並沿均線上行。近期KD在高檔鈍化後略有回落，但仍屬強勢整理；DIF線維持在MACD線之上，OSC

技術分析工具,解盤內容
K棒,近期先拉出一段連紅推升，逼近前高後出現回吐，K棒轉為紅黑交錯，顯示追價降溫、進入高檔消化；只要回檔不把前波起漲區跌破，仍偏向以整理取代反轉。
K線圖,整體節奏是底部抬高後的推進段，現階段卡在前高壓力帶下方震盪，較像「突破後拉回」的整理；若能守住52～53元區間並再度放量上攻，續攻機會較大，反之跌回52元下方，容易回到整理區反覆。
成交量,上攻到前高附近時量能有放大，回檔時量能相對收斂，屬於價漲量增、拉回量縮的健康型態；目前未見明顯價量背離，但若靠近55元仍量縮，突破力道會受限。
型態,目前走勢屬於雙重底，約在2025年10月20日至2026年3月20日形成，近期已回到頸線附近並在前高帶下方整理。
移動平均線,價格已站回5日線、10日線與20日線之上，短線拉回先看53元附近的5日線、10日線支撐；再來是52.0～52.5元附近的20日線支撐。上方54.5～55元為前高壓力帶，若突破後能站穩，才有機會帶動60日線走平翻揚、讓整理區間上移。
MACD指標,DIF線維持在MACD線之上，多方格局尚在；OSC柱體在翻正擴大後開始收斂，代表上攻動能轉為高檔消化，接下來要看柱體能否再度放大，配合價格突破55元壓力帶。
KD指標,K線與D線先前在高檔鈍化，近期K線開始回落並貼近D線，屬高檔降溫訊號；若回檔後能在中高檔再度向上拉開，會比較有利延續推升。
整體評價,偏多但在關鍵壓力前整理：55元附近是多空分水嶺，帶量突破並站穩，雙重底推進段可望延伸；操作上以52～53元為防守帶較合理，跌破52元則視為回到整理區，短線先轉保守。
